In [1]:
%load_ext autoreload
%autoreload 2

Failed to read module file 'C:\Users\Pavilion\AppData\Local\Programs\Python\Python311\Lib\shlex.py' for module 'shlex': UnicodeDecodeError
Traceback (most recent call last):
  File "f:\Desktop\DATA SCIENCE\Topics (Left)\MLOPs\CI-CD + Monitoring + Orchestration\Tutorial\monitoring\Lib\site-packages\IPython\core\extensions.py", line 62, in load_extension
    return self._load_extension(module_str)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "f:\Desktop\DATA SCIENCE\Topics (Left)\MLOPs\CI-CD + Monitoring + Orchestration\Tutorial\monitoring\Lib\site-packages\IPython\core\extensions.py", line 77, in _load_extension
    mod = import_module(module_str)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Pavilion\AppData\Local\Programs\Python\Python311\Lib\importlib\__init__.py", line 126, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1206, in 

In [2]:
%pwd

'f:\\Desktop\\DATA SCIENCE\\Topics (Left)\\MLOPs\\CI-CD + Monitoring + Orchestration\\Tutorial\\notebooks'

In [3]:
%cd ..

f:\Desktop\DATA SCIENCE\Topics (Left)\MLOPs\CI-CD + Monitoring + Orchestration\Tutorial


In [4]:
import os
import sys
import dill
import pandas as pd
import numpy as np
from mlops.exception import CustomException
from dataclasses import dataclass
from dataclasses import dataclass
from pathlib import Path
from mlops.constants import CONFIG_FILE_PATH
from mlops.utils.common import read_yaml, create_dir
from flask import Flask,request,render_template

In [5]:
from mlops.logger import setup_logger
logging = setup_logger()

In [6]:
# entity

@dataclass(frozen=True)
class PredictionConfig:
    preprocessor_path: Path
    model_path: Path

In [7]:
# configuration

class ConfigurationManager:
    def __init__(self, config_file_path=CONFIG_FILE_PATH):
        self.config = read_yaml(config_file_path)
        create_dir([self.config.artifacts_root])

    def get_prediction_config(self) -> PredictionConfig:
        return PredictionConfig(
            preprocessor_path=Path(self.config.data_transformation.preprocessor_obj_path),
            model_path=Path(self.config.training.trained_model_path)
        )

    

In [8]:
class PredictPipeline:
    def __init__(self):
        config = ConfigurationManager()
        pred_config = config.get_prediction_config()
        
        self.preprocessor = dill.load(pred_config.preprocessor_path)
        self.model = dill.load(pred_config.model_path)

        logging.info("model.dill and preprocessor.pkl objects are loaded successfully.")

    def predict(self, features):
        """This method will do the prediction."""
        try:
            logging.info("Starting prediction data preprocessing and prediction.")
            
            data_transformed = self.preprocessor.transform(features)
            preds = self.model.predict(data_transformed)

            logging.info("Finished prediction data transformatin and prediction.")

            return preds
        
        except Exception as e:
            raise CustomException(e,sys)

In [9]:
class CustomData:
    def __init__(  self,
        gender: str,
        race_ethnicity: str,
        parental_level_of_education,
        lunch: str,
        test_preparation_course: str,
        reading_score: int,
        writing_score: int):

        self.gender = gender

        self.race_ethnicity = race_ethnicity

        self.parental_level_of_education = parental_level_of_education

        self.lunch = lunch

        self.test_preparation_course = test_preparation_course

        self.reading_score = reading_score

        self.writing_score = writing_score

    def get_data_as_data_frame(self):
        try:
            custom_data_input_dict = {
                "gender": [self.gender],
                "race_ethnicity": [self.race_ethnicity],
                "parental_level_of_education": [self.parental_level_of_education],
                "lunch": [self.lunch],
                "test_preparation_course": [self.test_preparation_course],
                "reading_score": [self.reading_score],
                "writing_score": [self.writing_score],
            }

            return pd.DataFrame(custom_data_input_dict)

        except Exception as e:
            raise CustomException(e, sys)


In [ ]:
# from sklearn.preprocessing import StandardScaler
# from student_score_predictor.pipeline.predict_pipeline import CustomData,PredictPipeline

application=Flask(__name__)

app=application

## Route for a home page

@app.route('/')
def index():
    return render_template('index.html') 

@app.route('/predictdata',methods=['GET','POST'])
def predict_datapoint():
    if request.method=='GET':
        return render_template('home.html')
    else:
        data=CustomData(
            gender=request.form.get('gender'),
            race_ethnicity=request.form.get('ethnicity'),
            parental_level_of_education=request.form.get('parental_level_of_education'),
            lunch=request.form.get('lunch'),
            test_preparation_course=request.form.get('test_preparation_course'),
            reading_score=float(request.form.get('reading_score')),
            writing_score=float(request.form.get('writing_score'))

        )
        pred_df=data.get_data_as_data_frame()
        logging.info("prediction data is loaded successfully.")

        predict_pipeline=PredictPipeline()
        logging.info("Prediction starts --------->")

        results=predict_pipeline.predict(pred_df)
        logging.info("<--------- Prediction is completed successfully ------->")
        return render_template('home.html',results=results[0])
    

if __name__=="__main__":
    app.run(host="0.0.0.0", port=8000, debug=True)
        



 * Serving Flask app '__main__'
 * Debug mode: on


```python
   Got an exit=1 error
```